# Workshop 4 Lab — Intermediate
### Computation 101 · IQ Biology Bootcamp 2026

This is the last lab, and it changes the job. You already know how to write bash, use git, and analyze
a real dataset in Python. Today you point an **AI assistant** at that same work — and learn the one skill
that makes AI safe for science: **you verify everything it hands you against something you already know.**

The data is the same **ZFP36L2 knockout** you analyzed in Workshop 3, and that's the whole trick: *you own
the answers.* When the assistant writes code, you don't hope it's right — you check its number against the
number you derived yourself. **Run each cell in order** (Shift+Enter). Where you see `____`, fill the blank first.

### First: open an AI assistant in another tab

This lab is **tool-agnostic** — any chat assistant works, and they're all free. Pick one:

- **ChatGPT** — chat.openai.com
- **Claude** — claude.ai
- **Google Gemini** — gemini.google.com (or aistudio.google.com)
- **Colab's own Gemini** — the **✦** button in the toolbar, or the Gemini side panel, right here in this window. Free for all Colab users, so there's nothing to install and no key to paste.

You'll copy a prompt from this notebook, paste it into your assistant, then bring the answer back here and
**run it against the ground truth.** No API keys, no setup — you are the bridge between the two tabs.

In [ ]:
# One-time setup. The line starting with ! is a REAL shell command in this Colab Linux
# machine — the same git you used on Fiji. It clones the course repo (only if it isn't
# here yet) and points DATA at the ZFP36L2 dataset inside it.
![ -d iqbio-computation-101-2026 ] || git clone --depth 1 https://github.com/gsstephenson/iqbio-computation-101-2026.git
import pandas as pd, os
DATA = 'iqbio-computation-101-2026/data/zfp36l2'
print('data folder contains:', os.listdir(DATA))

## The dangerous kind of wrong
AI code that is *obviously* broken is harmless — it crashes, or the answer is absurd, and you throw it out.
The code that hurts your science is the code that **looks reasonable, runs cleanly, and is quietly wrong.**

Below are two functions an assistant produced for 'count the up-regulated genes in bone marrow.' Both run.
Both look fine. You know the true answer is **1343** — that's the only reason you're about to catch them.

## Bug 1 · the obvious-once-you-see-it one
Read this function. It looks like it filters on fold change and significance. Run it before you decide.

In [ ]:
def count_up_v1(path):
    df = pd.read_csv(path, index_col=0)
    # 'strong fold change in either direction, and significant'
    hits = df[(df['log2FoldChange'].abs() > 1) & (df['padj'] < 0.05)]
    return len(hits)

n = count_up_v1(os.path.join(DATA, 'de', 'BM_DE.csv'))
print('v1 says:', n, ' | you know it should be 1343 →', 'MATCH ✓' if n == 1343 else 'MISMATCH ✗')

Off by a lot. The bug is `.abs()`: **up**-regulated means `log2FoldChange > 1`, but `abs() > 1` also sweeps
in every strongly **down**-regulated gene (there are 846 of those in bone marrow). The assistant wrote
something plausible — 'strong change in either direction' — that quietly answers a *different question*.
Fix it: keep only genes going **up**.

In [ ]:
def count_up_fixed(path):
    df = pd.read_csv(path, index_col=0)
    hits = df[(df['log2FoldChange'] ____ 1) & (df['padj'] < 0.05)]   # up only
    return len(hits)

n = count_up_fixed(os.path.join(DATA, 'de', 'BM_DE.csv'))
print(n, '→', 'MATCH ✓' if n == 1343 else 'MISMATCH ✗')

## Bug 2 · the one you'd actually ship
This next function is more dangerous, because its answer is *close*. Run it and watch the number.

In [ ]:
def count_up_v2(path):
    df = pd.read_csv(path, index_col=0)
    # 'significant p-value and up-regulated'
    hits = df[(df['log2FoldChange'] > 1) & (df['pvalue'] < 0.05)]
    return len(hits)

n = count_up_v2(os.path.join(DATA, 'de', 'BM_DE.csv'))
print('v2 says:', n, ' | truth is 1343 →', 'MATCH ✓' if n == 1343 else 'MISMATCH ✗')

**1445, not 1343.** If you didn't already know the answer, would you have blinked at 1445? That's the point.

The bug is `pvalue` instead of `padj`. `pvalue` is the *raw, uncorrected* p-value. With ~17,000 genes tested
at once, a raw 0.05 cutoff lets a flood of false positives through — which is exactly why DESeq2 gives you
`padj`, the **multiple-testing-corrected** value. Swapping one column for a similarly-named one added **102
phantom genes** to your result. An assistant will make this substitution constantly; `padj` and `pvalue`
are both 'the p-value' to a language model. Fix it.

In [ ]:
def count_up_v2_fixed(path):
    df = pd.read_csv(path, index_col=0)
    hits = df[(df['log2FoldChange'] > 1) & (df['____'] < 0.05)]   # multiple-testing corrected
    return len(hits)

n = count_up_v2_fixed(os.path.join(DATA, 'de', 'BM_DE.csv'))
print(n, '→', 'MATCH ✓' if n == 1343 else 'MISMATCH ✗')

Two functions, both clean, both confidently wrong — one obvious, one subtle. The subtle one is the reason
'the code runs' will never again mean 'the code is right.' You caught both the same way: **against a number
you already trusted.** Next workbook: drive the assistant to *extend* the analysis, checking every step.

## One more thing

You just used AI the way a scientist has to — as a fast, confident junior colleague whose every claim you
check before you trust it. And you *could* check it, because the ground truth was real: a published study,
*The RNA binding protein ZFP36L2 displays tissue-selective mRNA targeting in mice*, **RNA Biology (2026)**.

Its first author is **George** — a first-year in your own program a year ago, sitting where you are now.
Across four workshops you took one real dataset from a raw shell prompt to a version-controlled, Python-analyzed,
AI-assisted result: **bash → git → Python → AI**, every step on the same science. The assistant made you faster.
Owning the answer is what kept you honest. That's the job now — and you can already do it.